# Verify: Do V11 clusters correspond to Left/Right movement variants?

We keep the FINE-GRAINED labels (UR, UL, OR, OL, USR, USL, OSR, OSL, FB, BF)
instead of merging them into 5 classes. Then check if K-means sub-clusters
align with L/R variants.

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Cycle extraction (same as V11 v2) ────────────────────────
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w','ay_w','az_w']].values
    omega = df[['gx','gy','gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag_deg = np.linalg.norm(omega, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7: return [], mag_deg, np.array([], dtype=int)
    win = int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if win % 2 == 0: win += 1
    win = max(5, min(win, n if n % 2 == 1 else n - 1))
    poly = max(1, min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']), win - 2))
    mag_s = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    mag_s = savgol_filter(mag_s, window_length=win, polyorder=poly, mode='interp')
    peaks, _ = find_peaks(mag_s, distance=max(1, int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)),
                          prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],
                          height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(peaks) == 0: return [], mag_s, peaks
    cycles = []
    for i, p in enumerate(peaks):
        left = 0 if i == 0 else (peaks[i-1] + p) // 2
        right = (len(t)-1) if i == len(peaks)-1 else (p + peaks[i+1]) // 2
        if right > left and (right - left) >= CONFIG['MIN_CYCLE_SAMPLES']:
            cycles.append((left, right))
    return cycles, mag_s, peaks

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T.astype(np.float32)

print('Functions defined')

In [ ]:
# ── FINE-GRAINED label mapping (keep L/R separate) ───────────
# Instead of merging UR/UL -> 'underhand', keep them distinct

FINE_MAP = {
    # Underhand variants
    'UR': 'UR', 'ur': 'UR', 'UR0': 'UR', 'UR-CW': 'UR',
    'UL0': 'UL',
    # Overhand variants
    'OR': 'OR', 'or': 'OR', 'OR2': 'OR', 'OR3': 'OR', 'OR-OSL': 'OR', 'OR/OSL?': 'OR',
    'OL': 'OL', 'ol': 'OL', 'OL2': 'OL',
    # Sneak underhand variants
    'USR': 'USR', 'USL': 'USL',
    # Sneak overhand variants
    'OSR': 'OSR', 'OSL': 'OSL',
    # Dragon roll variants
    'FB': 'FB', 'fb': 'FB', 'FB2': 'FB', 'BF2': 'BF', 'bf': 'BF',
    # Clean labels -> map to L/R where possible
    'underhand': 'U_generic', 'overhand': 'O_generic',
    'dragon_roll': 'DR_generic',
    'sneak_underhand': 'SU_generic', 'sneak_overhand': 'SO_generic',
    # Discard
    'CW': None, 'cw': None, 'CCW': None, 'ccw': None,
    'idle': None, 'Idle3': None, 'Idle-or-ol?': None,
    'VQ5': None, 'vq16': None,
}

# Also map by prefix for anything not in the exact map
_FINE_PREFIX = [
    (re.compile(r'^USR$', re.I), 'USR'), (re.compile(r'^USL$', re.I), 'USL'),
    (re.compile(r'^OSR$', re.I), 'OSR'), (re.compile(r'^OSL$', re.I), 'OSL'),
    (re.compile(r'^UR', re.I), 'UR'), (re.compile(r'^UL', re.I), 'UL'),
    (re.compile(r'^OR', re.I), 'OR'), (re.compile(r'^OL', re.I), 'OL'),
    (re.compile(r'^FB', re.I), 'FB'), (re.compile(r'^BF', re.I), 'BF'),
    (re.compile(r'^CW$', re.I), None), (re.compile(r'^CCW$', re.I), None),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]

def map_fine_label(raw):
    raw = raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for pat, c in _FINE_PREFIX:
        if pat.match(raw): return c
    return None

# Coarse mapping (for comparison)
COARSE_MAP = {
    'UR': 'underhand', 'UL': 'underhand', 'U_generic': 'underhand',
    'OR': 'overhand', 'OL': 'overhand', 'O_generic': 'overhand',
    'USR': 'sneak_underhand', 'USL': 'sneak_underhand', 'SU_generic': 'sneak_underhand',
    'OSR': 'sneak_overhand', 'OSL': 'sneak_overhand', 'SO_generic': 'sneak_overhand',
    'FB': 'dragon_roll', 'BF': 'dragon_roll', 'DR_generic': 'dragon_roll',
}

KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}

print('Fine-grained label mapping defined')
print('Fine classes:', sorted(set(v for v in FINE_MAP.values() if v is not None)))

In [ ]:
# ── Load all cycles with FINE labels ────────────────────────

all_matrices = []; all_fine_labels = []; all_coarse_labels = []; all_groups = []

def load_cycles(d0_path, d1_path, name, fine_label='unlabeled', windows=None):
    df0, df1 = pd.read_csv(d0_path), pd.read_csv(d1_path)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    cyc0, _, _ = detect_cycles(t0, om0, CONFIG['FS'])
    cyc1, _, _ = detect_cycles(t1, om1, CONFIG['FS'])
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if windows is not None:
        fp0, fp1 = [], []
        for (s0,e0),(s1,e1) in zip(p0,p1):
            t_mid = (t0[s0]+t0[e0])/2
            if any(ws <= t_mid < we for ws, we in windows):
                fp0.append((s0,e0)); fp1.append((s1,e1))
        p0, p1 = fp0, fp1
    grp = name.split('/')[0] if '/' in name else name
    count = 0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm)
        all_fine_labels.append(fine_label)
        coarse = COARSE_MAP.get(fine_label, fine_label)
        all_coarse_labels.append(coarse)
        all_groups.append(grp)
        count += 1
    return count

# Homogeneous (these have coarse labels only — generic)
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv'))):
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    fine_label = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            if pat == 'underhand_default': fine_label = 'U_generic'
            elif pat == 'underhand': fine_label = 'U_generic'
            elif pat == 'overhand': fine_label = 'O_generic'
            elif pat == 'dragon_roll': fine_label = 'DR_generic'
            elif pat == 'sneak_underhand': fine_label = 'SU_generic'
            elif pat == 'sneak_overhand': fine_label = 'SO_generic'
            else: fine_label = pat
            break
    n = load_cycles(d0, d1, stem, fine_label)
    if n > 0: print(f'  {stem:<50s} {fine_label:<15s} {n:>4d}')

# Heterogeneous (these have FINE labels — UR, UL, OR, OL, etc.)
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f: data = json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            raw_label = seg.get('label', '')
            fine = map_fine_label(raw_label)
            if fine is None: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[fine].append((float(s), float(e)))
        for fine_label, windows in sorted(wbl.items()):
            n = load_cycles(d0, d1, sname + '/' + fine_label, fine_label, windows)
            if n > 0: print(f'  {sname}/{fine_label:<15s} {"":>29s} {n:>4d}')

X_all = np.array([cm.ravel() for cm in all_matrices])
fine_all = np.array(all_fine_labels)
coarse_all = np.array(all_coarse_labels)
g_all = np.array(all_groups)

labeled_mask = np.array([l != 'unlabeled' for l in fine_all])
# Fine labels that have L/R info (not generic)
lr_mask = np.array([l in ('UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF') for l in fine_all])

print('\nTotal: ' + str(len(X_all)) + ' cycles')
print('Labeled (any): ' + str(labeled_mask.sum()))
print('L/R labeled: ' + str(lr_mask.sum()))
print('\nFine label distribution:')
for lab, cnt in sorted(Counter(fine_all[labeled_mask]).items(), key=lambda x: -x[1]):
    print(f'  {lab:<15s}: {cnt:>5d}')

In [ ]:
# ── PCA + K-means k=12 (same as V11 v2) ──────────────────────

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
pca = PCA(n_components=min(50, X_scaled.shape[1], X_scaled.shape[0]-1), svd_solver='full')
X_pca = pca.fit_transform(X_scaled)

km = KMeans(n_clusters=12, random_state=42, n_init=20, max_iter=500)
cluster_ids = km.fit_predict(X_pca)

print('K-means k=12 done on ' + str(X_pca.shape[1]) + 'D PCA')

# t-SNE for visualization
print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca)
print('t-SNE done')

In [ ]:
# ── KEY ANALYSIS: Cross-tab clusters vs fine L/R labels ──────

# Only look at cycles that have L/R labels
print('CLUSTER vs FINE L/R LABELS (only L/R-labeled cycles)')
print('='*80)

lr_fine = fine_all[lr_mask]
lr_clusters = cluster_ids[lr_mask]

# Cross-tabulation
fine_classes = sorted(set(lr_fine))
print(f'\nFine L/R classes: {fine_classes}')
print(f'L/R labeled cycles: {lr_mask.sum()}')

# For each cluster, show fine label breakdown
print(f'\n{"Cluster":>8s}  {"Size":>6s}  ', end='')
for fc in fine_classes:
    print(f'{fc:>6s}', end=' ')
print('  Dominant    Split?')
print('-' * (30 + 7*len(fine_classes)))

for c in range(12):
    mask = lr_clusters == c
    size = mask.sum()
    if size == 0: continue
    counts = Counter(lr_fine[mask])
    print(f'  C{c:>2d}    {size:>5d}  ', end='')
    for fc in fine_classes:
        print(f'{counts.get(fc, 0):>6d}', end=' ')
    dom = counts.most_common(1)[0][0] if counts else '-'
    # Check if this cluster separates L from R within a movement
    # e.g., has mostly OR but not OL, or mostly UR but not UL
    has_R = sum(counts.get(x, 0) for x in ['UR','OR','USR','OSR'])
    has_L = sum(counts.get(x, 0) for x in ['UL','OL','USL','OSL'])
    has_FB = sum(counts.get(x, 0) for x in ['FB','BF'])
    total_lr = has_R + has_L
    if total_lr > 5:
        r_frac = has_R / total_lr
        if r_frac > 0.7:
            split = 'RIGHT-dominant'
        elif r_frac < 0.3:
            split = 'LEFT-dominant'
        else:
            split = 'mixed L/R'
    elif has_FB > 5:
        split = 'dragon_roll'
    else:
        split = '-'
    print(f'  {dom:<10s}  {split}')

print()

In [ ]:
# ── Visualization: t-SNE colored by fine L/R labels ──────────

FINE_COLORS = {
    'UR': '#2ecc71', 'UL': '#27ae60',  # underhand: green shades
    'OR': '#9b59b6', 'OL': '#8e44ad',  # overhand: purple shades
    'USR': '#f39c12', 'USL': '#e67e22',  # sneak underhand: orange
    'OSR': '#e74c3c', 'OSL': '#c0392b',  # sneak overhand: red
    'FB': '#3498db', 'BF': '#2980b9',  # dragon roll: blue
    'U_generic': '#5DCAA5', 'O_generic': '#7F77DD',
    'DR_generic': '#E24B4A', 'SU_generic': '#EF9F27', 'SO_generic': '#D85A30',
    'unlabeled': '#DDDDDD',
}

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# Left: colored by fine L/R labels
ax = axes[0]
ul = fine_all == 'unlabeled'
ax.scatter(X_tsne[ul, 0], X_tsne[ul, 1], c='#EEEEEE', s=3, alpha=0.2)
# Plot generic labels
for lab in ['U_generic', 'O_generic', 'DR_generic', 'SU_generic', 'SO_generic']:
    m = fine_all == lab
    if m.sum() > 0:
        ax.scatter(X_tsne[m, 0], X_tsne[m, 1], c=FINE_COLORS.get(lab, '#888'),
                   s=10, alpha=0.4, label=lab, marker='o')
# Plot L/R labels (on top, larger, distinct markers)
for lab in ['UR', 'UL', 'OR', 'OL', 'USR', 'USL', 'OSR', 'OSL', 'FB', 'BF']:
    m = fine_all == lab
    if m.sum() > 0:
        marker = 'v' if 'R' in lab or lab == 'FB' else '^'  # R=down triangle, L=up triangle
        ax.scatter(X_tsne[m, 0], X_tsne[m, 1], c=FINE_COLORS.get(lab, '#888'),
                   s=25, alpha=0.8, label=lab, marker=marker, edgecolors='black', linewidth=0.3)
ax.legend(fontsize=7, ncol=2, loc='best')
ax.set_title('t-SNE colored by FINE L/R labels', fontsize=13)

# Right: colored by K-means clusters
ax = axes[1]
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_ids, cmap='tab20', s=5, alpha=0.5)
ax.set_title('t-SNE colored by K-means k=12', fontsize=13)

plt.tight_layout()
plt.savefig('v11_lr_verification.png', dpi=150)
plt.show()

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────

print('='*70)
print('V11 L/R VERIFICATION SUMMARY')
print('='*70)

print(f'Total cycles: {len(X_all)}')
print(f'Labeled (any): {labeled_mask.sum()}')
print(f'L/R fine-labeled: {lr_mask.sum()}')
print(f'Generic-labeled: {labeled_mask.sum() - lr_mask.sum()}')
print(f'Unlabeled: {(~labeled_mask).sum()}')

print(f'\nFine label counts:')
for lab, cnt in sorted(Counter(fine_all[lr_mask]).items(), key=lambda x: -x[1]):
    print(f'  {lab:<6s}: {cnt:>5d}')

# Key question: do L and R separate into different clusters?
print(f'\nL/R SEPARATION ANALYSIS:')
movements = [('underhand', 'UR', 'UL'), ('overhand', 'OR', 'OL'),
             ('sneak_underhand', 'USR', 'USL'), ('sneak_overhand', 'OSR', 'OSL')]
for name, right, left in movements:
    r_mask = fine_all == right
    l_mask = fine_all == left
    if r_mask.sum() == 0 and l_mask.sum() == 0:
        print(f'  {name}: no L/R data')
        continue
    r_clusters = set(cluster_ids[r_mask]) if r_mask.sum() > 0 else set()
    l_clusters = set(cluster_ids[l_mask]) if l_mask.sum() > 0 else set()
    overlap = r_clusters & l_clusters
    r_only = r_clusters - l_clusters
    l_only = l_clusters - r_clusters
    # Compute how often L and R land in DIFFERENT clusters
    if r_mask.sum() > 0 and l_mask.sum() > 0:
        r_dist = Counter(cluster_ids[r_mask])
        l_dist = Counter(cluster_ids[l_mask])
        # Overlap coefficient
        shared_count = sum(min(r_dist.get(c, 0), l_dist.get(c, 0)) for c in range(12))
        total = r_mask.sum() + l_mask.sum()
        separation = 1 - (2 * shared_count / total)
    else:
        separation = -1
    print(f'  {name}:')
    print(f'    {right}: {r_mask.sum()} cycles in clusters {sorted(r_clusters)}')
    print(f'    {left}: {l_mask.sum()} cycles in clusters {sorted(l_clusters)}')
    print(f'    Shared clusters: {sorted(overlap)}')
    print(f'    {right}-only clusters: {sorted(r_only)}')
    print(f'    {left}-only clusters: {sorted(l_only)}')
    if separation >= 0:
        print(f'    Separation score: {separation:.2f} (1.0=fully separate, 0.0=fully mixed)')

# Dragon roll
fb_mask = fine_all == 'FB'
bf_mask = fine_all == 'BF'
if fb_mask.sum() > 0 or bf_mask.sum() > 0:
    print(f'  dragon_roll:')
    print(f'    FB: {fb_mask.sum()} cycles in clusters {sorted(set(cluster_ids[fb_mask]))}')
    print(f'    BF: {bf_mask.sum()} cycles in clusters {sorted(set(cluster_ids[bf_mask]))}')

print(f'\nNMI (fine L/R labels vs clusters): {normalized_mutual_info_score(fine_all[lr_mask], cluster_ids[lr_mask]):.3f}')
print(f'NMI (coarse 5-class vs clusters): {normalized_mutual_info_score(coarse_all[labeled_mask], cluster_ids[labeled_mask]):.3f}')